In [1]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from mindspore.dataset import audio

target_dir = Path("./archive")
all_wavs = list(target_dir.rglob("*.wav"))
train_set, test_set = train_test_split(all_wavs, test_size=0.4, shuffle=True)
val_set, test_set = train_test_split(test_set, test_size=0.5, shuffle=True)


In [2]:
from scipy.io import wavfile
samplerate, data = wavfile.read(train_set[0])
data

array([ 10,  14,  12, ..., -25, -22, -20], dtype=int16)

In [3]:
all_classes = []

for i in range(len(val_set)):
    val = str(val_set[i].parent).split("/")[-1]
    all_classes.append(val)

for i in range(len(train_set)):
    val = str(train_set[i].parent).split("/")[-1]
    all_classes.append(val)

for i in range(len(test_set)):
    val = str(test_set[i].parent).split("/")[-1]
    all_classes.append(val)

all_classes = list(set(all_classes))
all_classes

['five',
 'four',
 'stop',
 'wow',
 'right',
 'bed',
 'no',
 'on',
 'sheila',
 'marvin',
 'eight',
 'bird',
 'dog',
 'up',
 'house',
 'cat',
 'zero',
 'seven',
 'three',
 'yes',
 'left',
 'two',
 'happy',
 'six',
 'down',
 'one',
 'off',
 'tree',
 'nine',
 'go']

In [4]:
import numpy as np

np.ndarray.tolist

<method 'tolist' of 'numpy.ndarray' objects>

In [5]:
import mindspore
from mindspore.dataset import transforms

class AudioDataset:
    def __init__(self, audio_files, all_classes, max_len=80):
        self.audio_files = audio_files
        self.all_classes = all_classes
        self.max_len = max_len

    def __len__(self):
        return len(self.audio_files)
    
    def __getitem__(self, idx):
            audio_path = self.audio_files[idx]
            class_name = str(audio_path.parent).split("/")[-1]
            label = self.all_classes.index(class_name)
            one_hot = mindspore.Tensor(mindspore.numpy.zeros((len(self.all_classes))), dtype=mindspore.float32)
            one_hot[label] = 1.0
            _, data = wavfile.read(audio_path)
            return data, one_hot


def resize_axis(arr, max_len=80):
    arr = arr[:, :max_len]
    current_length = arr.shape[1]
    pad_amount = max_len - current_length
    if pad_amount > 0:
        padded_arr = np.pad(arr, ((0, 0), (0, pad_amount)), mode='constant', constant_values=0)
    else:
        padded_arr = arr       
    return padded_arr

def add_channel(arr):    
    return arr[np.newaxis, :]

def create_mindspore_dataset(audio_files, all_classes, workers=1, batch_size=32, max_len=80):
    audio_ds = AudioDataset(audio_files=audio_files, all_classes=all_classes)
    data_set = mindspore.dataset.GeneratorDataset(source=audio_ds, column_names=["audio", "label"])

    trans = [
        audio.MelSpectrogram(n_mels=64),
        (lambda x: resize_axis(x, max_len)),
        (lambda x: add_channel(x)), 
        transforms.TypeCast(mindspore.float32)
    ]

    target_trans = transforms.TypeCast(mindspore.float32)

    data_set = data_set.map(
        operations=trans,
        input_columns='audio',
        num_parallel_workers=workers
    )

    data_set = data_set.map(
        operations=target_trans,
        input_columns='label',
        num_parallel_workers=workers
    )

    data_set = data_set.batch(batch_size, drop_remainder=True)
    return data_set

In [6]:
mindspore_train = create_mindspore_dataset(train_set, all_classes)
mindspore_val = create_mindspore_dataset(val_set, all_classes)
mindspore_test = create_mindspore_dataset(test_set, all_classes)

[WARNING] ME(49732:137639808382080,MainProcess):2026-03-26-18:20:53.356.000 [mindspore/dataset/engine/datasets_user_defined.py:1042] Input 'source' of 'GeneratorDataset' includes network computing operators like in mindspore.nn, mindspore.ops, mindspore.numpy module and etc, which do not support multi-thread compiling, recommend to replace it with python implemented operator like numpy etc. Here decrease 'num_parallel_workers' into 1.
[WARNING] ME(49732:137639808382080,MainProcess):2026-03-26-18:20:53.358.000 [mindspore/dataset/engine/datasets.py:1039] Input 'operations' of 'map' includes network computing operators like in mindspore.nn, mindspore.ops, mindspore.numpy module and etc, which do not support multithreading compiling, recommend to replace it with python implemented operator like numpy etc. Here decrease 'num_parallel_workers' into 1.
[WARNING] ME(49732:137639808382080,MainProcess):2026-03-26-18:20:53.360.000 [mindspore/dataset/engine/datasets_user_defined.py:1042] Input 'so

In [7]:
train_dict = mindspore_train.create_dict_iterator()
for batch in train_dict:
    input_shape = batch["audio"].shape
    label_shape = batch["label"].shape
    print(input_shape)
    print(label_shape)
    break

(32, 1, 64, 80)
(32, 30)


In [8]:
import mindspore.nn as nn

class AudioModel(nn.Cell):
    def __init__(self, dummy_input_shape, convolutional_features, dense_features, num_classes):
        super(AudioModel, self).__init__()
        layers = []
        in_features = dummy_input_shape[1]
        for out_features in convolutional_features:
            layers.append(nn.Conv2d(in_channels=in_features,
                                    out_channels=out_features,
                                    kernel_size=3,
                                    stride=1,
                                    pad_mode="valid"))
            layers.append(nn.BatchNorm2d(num_features=out_features))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(kernel_size=3, stride=2))
            in_features = out_features

        self.cnn = nn.SequentialCell(layers)
        self.flatten = nn.Flatten()
        in_dense = self._get_flat_dims(dummy_input_shape)
        dense_layers = []
        for out_dense in dense_features:
            dense_layers.append(nn.Linear(in_dense, out_dense))
            dense_layers.append(nn.ReLU())
            dense_layers.append(nn.Dropout(p=0.2))
            in_dense = out_dense
        dense_layers.append(nn.Linear(out_dense, num_classes))
        self.dense = nn.SequentialCell(dense_layers)

    def _get_flat_dims(self, dummy_input_shape):
        all_zeros = mindspore.numpy.zeros(dummy_input_shape)
        out = self.cnn(all_zeros)
        return self.flatten(out).shape[-1]
    
    def construct(self, x):
        out = self.cnn(x)
        out = self.flatten(out)
        out = self.dense(out)
        return out
    

conv_features = [32, 64, 128, 256]
dense_features = [512, 256, 128]
audio_model = AudioModel(input_shape, conv_features, dense_features, num_classes=label_shape[-1])
print(audio_model)

AudioModel(
  (cnn): SequentialCell(
    (0): Conv2d(input_channels=1, output_channels=32, kernel_size=(3, 3), stride=(1, 1), pad_mode=valid, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0x7d2ebc1205e0>, bias_init=None, format=NCHW)
    (1): BatchNorm2d(num_features=32, eps=1e-05, momentum=0.9, gamma=Parameter (name=cnn.1.gamma, shape=(32,), dtype=Float32, requires_grad=True), beta=Parameter (name=cnn.1.beta, shape=(32,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=cnn.1.moving_mean, shape=(32,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=cnn.1.moving_variance, shape=(32,), dtype=Float32, requires_grad=False))
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=2, pad_mode=VALID)
    (4): Conv2d(input_channels=32, output_channels=64, kernel_size=(3, 3), stride=(1, 1), pad_mode=valid, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.i

In [9]:
for param in audio_model.get_parameters():
    param.requires_grad = True

In [ ]:
from tqdm import tqdm

train_dict = mindspore_train.create_dict_iterator()
val_dict = mindspore_val.create_dict_iterator()
test_dict = mindspore_test.create_dict_iterator()

best_model_path = "./audio-model-best.ckpt"
best_val_acc = 0


optimizer = nn.AdamWeightDecay(audio_model.trainable_params(), learning_rate=0.0002)
loss_fn = nn.CrossEntropyLoss()

epochs = 100

def forward_fn(data, label):
    logits = audio_model(data)
    loss = loss_fn(logits, label)
    return loss, logits

grad_fn = mindspore.ops.value_and_grad(forward_fn, None, optimizer.parameters, has_aux=True)

for epoch in range(epochs):
    total_loss = 0
    total_accuracy = 0
    num_steps = 0
    audio_model.set_train(True)
    for batch in tqdm(train_dict):
        x = batch["audio"]
        y = batch["label"]
        (loss, logits), grads = grad_fn(x, y)
        optimizer(grads)
        total_loss += loss.numpy()
        predicted_class = y.numpy().argmax(axis=1)
        true_class = logits.numpy().argmax(axis=1)
        acc = np.sum(predicted_class == true_class)
        total_accuracy += acc
        num_steps += x.shape[0]
    print(f"Train Epoch: {epoch} Loss: {total_loss/num_steps} Accuracy: {total_accuracy/num_steps}")

    val_accuracy = 0
    num_val_steps = 0
    audio_model.set_train(False)
    for batch in tqdm(val_dict):
        x = batch["audio"]
        y = batch["label"]
        logits = audio_model(x)
        predicted_val = y.numpy().argmax(axis=1)
        true_val = logits.numpy().argmax(axis=1)
        val_acc = np.sum(predicted_val == true_val)
        val_accuracy += val_acc
        num_val_steps += x.shape[0]
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        mindspore.save_checkpoint(audio_model, best_model_path)
    print(f"Validation Epoch: {epoch} Accuracy: {val_accuracy/num_val_steps}")

0it [00:00, ?it/s]

975it [05:49,  2.56it/s]

In [ ]:
del audio_model

In [ ]:
conv_features = [32, 64, 128, 256]
dense_features = [512, 256, 128]
audio_model = AudioModel(input_shape, conv_features, dense_features, num_classes=label_shape[-1])
print(audio_model)

audio_params = mindspore.load_checkpoint(best_model_path)
mindspore.load_obf_params_into_net(audio_model, audio_params)
audio_model.set_train(False)

test_accuracy = 0
num_test_steps = 0
audio_model.set_train(False)
for batch in tqdm(test_dict):
    x = batch["audio"]
    y = batch["label"]
    logits = audio_model(x)
    predicted_test = y.numpy().argmax(axis=1)
    true_test = logits.numpy().argmax(axis=1)
    test_acc = np.sum(predicted_test == true_test)
    test_accuracy += test_acc
    num_test_steps += x.shape[0]
print(f"Validation Epoch: {epoch} Accuracy: {test_accuracy/num_test_steps}")